# Revenue Data and Building a Dashboard
## Extracting and Visualizing Stock Data
This assignment extracts Tesla and GameStop stock data, collects quarterly revenue data using web scraping, and visualizes stock price and revenue.

**Estimated time:** 30 minutes


## Setup
Run this cell if the required packages are not already installed in your Jupyter environment.

In [ ]:
%pip install yfinance beautifulsoup4 nbformat matplotlib pandas requests

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

## Define a Function that Makes a Graph
The function below takes stock data containing `Date` and `Close` columns, revenue data containing `Date` and `Revenue` columns, and the stock name. It displays stock price and revenue up to the dates specified in the assignment.

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(pd.to_datetime(stock_data_specific.Date),
                 stock_data_specific.Close.astype(float), label='Share Price')
    axes[0].set_ylabel('Price ($US)')
    axes[0].set_title(f'{stock} - Historical Share Price')

    axes[1].plot(pd.to_datetime(revenue_data_specific.Date),
                 revenue_data_specific.Revenue.astype(float), label='Revenue')
    axes[1].set_ylabel('Revenue ($US Millions)')
    axes[1].set_xlabel('Date')
    axes[1].set_title(f'{stock} - Historical Revenue')
    plt.tight_layout()
    plt.show()

## Question 1: Use yfinance to Extract Tesla Stock Data
Use the ticker symbol `TSLA`. Extract the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
tesla = yf.Ticker('TSLA')
tesla_data = tesla.history(period='max')
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2: Use Web Scraping to Extract Tesla Revenue Data
Download the Tesla revenue webpage, parse it with BeautifulSoup, extract the quarterly revenue table, and clean the Revenue column.

In [ ]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm'
html_data = requests.get(url).text
soup = BeautifulSoup(html_data, 'html.parser')

tesla_revenue = pd.DataFrame(columns=['Date', 'Revenue'])
for row in soup.find_all('tbody')[1].find_all('tr'):
    col = row.find_all('td')
    if len(col) >= 2:
        date = col[0].text
        revenue = col[1].text
        tesla_revenue = pd.concat(
            [tesla_revenue, pd.DataFrame({'Date': [date], 'Revenue': [revenue]})],
            ignore_index=True
        )

tesla_revenue['Revenue'] = tesla_revenue['Revenue'].str.replace(',|\\$', '', regex=True)
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != '']
tesla_revenue.tail()

## Question 3: Use yfinance to Extract GameStop Stock Data
Use the ticker symbol `GME`. Extract the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
gme = yf.Ticker('GME')
gme_data = gme.history(period='max')
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4: Use Web Scraping to Extract GameStop Revenue Data
Download the GameStop revenue webpage, parse it with BeautifulSoup, extract the quarterly revenue table, clean the Revenue column, and display the last five rows.

In [ ]:
url_2 = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html'
html_data_2 = requests.get(url_2).text
soup_2 = BeautifulSoup(html_data_2, 'html.parser')

gme_revenue = pd.DataFrame(columns=['Date', 'Revenue'])
for row in soup_2.find_all('tbody')[1].find_all('tr'):
    col = row.find_all('td')
    if len(col) >= 2:
        gme_revenue = pd.concat(
            [gme_revenue, pd.DataFrame({'Date': [col[0].text], 'Revenue': [col[1].text]})],
            ignore_index=True
        )

gme_revenue['Revenue'] = gme_revenue['Revenue'].str.replace(',', '').str.replace('$', '')
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != '']
gme_revenue.tail()

## Question 5: Plot Tesla Stock Graph
Use the `make_graph` function to graph Tesla stock data and revenue.

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6: Plot GameStop Stock Graph
Use the `make_graph` function to graph GameStop stock data and revenue.

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')

## Submission Note
Run the notebook cells from top to bottom. Save the completed notebook as an `.ipynb` file and upload that file to the assignment portal. The original PDF asks for screenshots of selected code and outputs; include those separately if your instructor or portal requires them.